# GUV Fluorescence Analysis Pipeline

**Workflow:**
1. Load a VSI file — Series 0 (GFP + mCherry) only; BF and macro series are ignored
2. Split channels by explicit index
3. Run Cellpose on GFP to segment GUVs
4. **Inspect & edit masks interactively in napari**
5. Derive membrane ring + lumen masks
6. Quantify fluorescence in both channels
7. Batch-process remaining files

---
### Installation (run once)
```bash
conda create -n guv_env python=3.10
conda activate guv_env
pip install aicsimageio[bioformats] bioformats_jar cellpose "napari[all]" pyqt5 \
            numpy scipy pandas matplotlib seaborn notebook ipykernel
python -m ipykernel install --user --name guv_env --display-name "GUV pipeline"
# Java JDK 8+ must be on PATH for BioFormats VSI reading
# Xcode Command Line Tools required on macOS: xcode-select --install

from cellpose import models
model = models.CellposeModel(pretrained_model='cpsam') ## This will trigger the download of cpsam model — can take a few minutes, ~400MB
```

> **VS Code users:** napari opens as a separate desktop window via `napari.run()`. Edit masks there, close the window, then run the retrieval cell to continue. Do NOT use `%gui qt`.

### 0a. Import libs

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import ndimage as ndi
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from aicsimageio import AICSImage
from aicsimageio.readers.bioformats_reader import BioformatsReader

import tifffile

from cellpose import models
import napari

print('Imports OK')

---
### 0b · Configuration — edit these paths & parameters

In [ ]:
# ── PATHS ──────────────────────────────────────────────────────────────────────
INPUT_DIR  = Path('')          # folder containing .vsi files
OUTPUT_DIR = Path('') # will be created if absent
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── CHANNEL INDICES within Series 0 ───────────────────────────────────────────
# S=0 contains exactly 2 planes: GFP and mCherry.
# BF (S=1) and macro image (S=2) are separate series and are never loaded.
# Run the sanity-check cell below first if you are unsure of the order.
GFP_CHANNEL_IDX     = 0   # first plane  of S=0
MCHERRY_CHANNEL_IDX = 1   # second plane of S=0

# ── SEGMENTATION CHANNEL ─────────────────────────────────────────────────────
SEGMENTATION_CHANNEL_IDX = GFP_CHANNEL_IDX   # GFP_CHANNEL_IDX or MCHERRY_CHANNEL_IDX

# ── CELLPOSE PARAMETERS ────────────────────────────────────────────────────────
SPATIAL_RESOLUTION_PX_UM = 15.3846
MINIMUM_PHYSICAL_DIAMETER_UM = 8.0 # minimal diameter for detection
CP_MIN_DIAMETER_PX = MINIMUM_PHYSICAL_DIAMETER_UM * SPATIAL_RESOLUTION_PX_UM
CP_MIN_AREA_PX = int(np.pi * (CP_MIN_DIAMETER_PX / 2.0)**2)
CP_DIAMETER           = None    # px -- None = Cellpose auto-estimates
CP_FLOW_THRESHOLD     = 0     # lower = keep more objects
CP_CELLPROB_THRESHOLD = 0    # lower = detect dimmer GUVs (try -3 or -4)
CP_MODEL              = 'cyto3' # 'cyto3' | 'cyto2'
CP_GPU                = True   # True if CUDA is available

# ── QUANTIFICATION ─────────────────────────────────────────────────────────────
MEMBRANE_THICKNESS_PX = 7       # pixels -- increase if membrane appears thicker

# List all VSI files found
vsi_files = sorted(INPUT_DIR.glob('*.vsi'))
print(f'Found {len(vsi_files)} VSI file(s):')
for f in vsi_files:
    print(f'  {f.name}')

### 1 · Imports & helper functions

In [ ]:
# ── Core pipeline functions ────────────────────────────────────────────────────

def read_vsi(vsi_path):
    """
    Read a VSI file with aicsimageio + BioFormats.

    CellSens VSI series structure:
      S=0  GFP, mCherry  (2048x2048, 2 planes)  <- we load this
      S=1  BF            (2048x2048, 1 plane)    <- ignored
      S=2  macro image   (512x512,   3 planes)   <- ignored

    Returns: (data CYX array, channel_names list)
    """

    img = AICSImage(str(vsi_path),reader=BioformatsReader)

    # Print all series so unexpected structures are immediately visible
    print(f'  File      : {Path(vsi_path).name}')
    print(f'  All series: {img.scenes}')

    # Always load Series 0 - the fluorescence series
    data = img.get_image_data('CYX', S=0, T=0, Z=0)

    # Safety check - fail loudly if the series structure is unexpected
    assert data.shape[0] == 2, (
        f"Expected exactly 2 channels in S=0 (GFP + mCherry), "
        f"got {data.shape[0]}.\n"
        f"Check the series list printed above and adjust S= or the channel indices."
    )

    print(f'  S=0 shape : {data.shape}  dtype={data.dtype}')
    print(f'  Channel names in S=0: {img.channel_names}')
    return data, img.channel_names


def split_channels(data, channel_names, gfp_idx=0, mcherry_idx=1):
    """
    Split by explicit channel index.
    Prints a full channel listing so you can verify the assignment every time.
    """
    print(f'  Channels in this file:')
    for i, name in enumerate(channel_names):
        tag = ''
        if i == gfp_idx:     tag = '  <- GFP'
        if i == mcherry_idx: tag = '  <- mCherry'
        print(f'    [{i}] "{name}"{tag}')
    return data[gfp_idx], data[mcherry_idx]


def normalise(img):
    """Percentile stretch to 0-255 float32 (for Cellpose & display)."""
    lo, hi = np.percentile(img, (1, 99))
    return np.clip((img.astype(np.float32) - lo) / (hi - lo + 1e-9), 0, 1) * 255.0


def segment_guvs(gfp_img, diameter=None, flow_threshold=0.4,
                 cellprob_threshold=-2.0, model_type='cyto3', use_gpu=False,
                 min_size=15):
    """Run Cellpose on GFP channel -> integer label mask (0=background)."""
    
    model = models.CellposeModel(gpu=use_gpu, pretrained_model=model_type)
    
    # 1. Unpack exactly three values (masks, flows, styles)
    masks, flows, styles = model.eval(
        normalise(gfp_img),
        diameter=diameter,
        min_size=min_size,
        flow_threshold=flow_threshold,
        cellprob_threshold=cellprob_threshold,
        do_3D=False,
    )

    n_guvs = int(masks.max())
    if diameter is None:
        _labels = np.unique(masks)
        _labels = _labels[_labels != 0]
        if _labels.size:
            _areas   = np.array([(masks == lbl).sum() for lbl in _labels])
            _diam_est = float(np.mean(2.0 * np.sqrt(_areas / np.pi)))
            print(f'  Cellpose -> {n_guvs} GUVs detected  '
                  f'(mean diameter from masks: {_diam_est:.1f} px — '
                  f'set CP_DIAMETER to this value to skip auto-estimation)')
        else:
            print(f'  Cellpose -> 0 GUVs detected')
    else:
        print(f'  Cellpose -> {n_guvs} GUVs detected')
    
    return masks.astype(np.int32)


def make_membrane_lumen_masks(masks, membrane_thickness_px=5):
    """Erode each GUV label to produce membrane-ring and lumen dicts."""
    
    struct = ndi.generate_binary_structure(2, 1)
    membrane_masks, lumen_masks = {}, {}
    
    active_labels = np.unique(masks)
    active_labels = active_labels[active_labels != 0]
    for label in active_labels:
        binary = masks == label
        eroded = binary.copy()
        for _ in range(membrane_thickness_px):
            eroded = ndi.binary_erosion(eroded, structure=struct)
            
        if not eroded.any():
            membrane_masks[label] = binary
            lumen_masks[label]    = np.zeros_like(binary)
        else:
            membrane_masks[label] = binary & ~eroded
            lumen_masks[label]    = eroded
            
    return membrane_masks, lumen_masks


def quantify(image, masks, membrane_masks, lumen_masks, channel_name):
    """Measure mean/integrated intensity in membrane and lumen per GUV."""
    rows = []
    for label in membrane_masks:
        m_px = image[membrane_masks[label]]
        l_px = image[lumen_masks[label]]
        full = masks == label
        ys, xs = np.where(full)
        row = {
            'label': label,
            'centroid_x_px': float(xs.mean()),
            'centroid_y_px': float(ys.mean()),
            'area_px2': int(full.sum()),
            'diameter_px': 2.0 * np.sqrt(full.sum() / np.pi),
            f'{channel_name}_membrane_mean':       float(m_px.mean()) if m_px.size else np.nan,
            f'{channel_name}_membrane_integrated': float(m_px.sum())  if m_px.size else np.nan,
            f'{channel_name}_lumen_mean':          float(l_px.mean()) if l_px.size else np.nan,
            f'{channel_name}_lumen_integrated':    float(l_px.sum())  if l_px.size else np.nan,
        }
        denom = row[f'{channel_name}_lumen_mean']
        row[f'{channel_name}_mem_lumen_ratio'] = (
            row[f'{channel_name}_membrane_mean'] / (denom + 1e-9)
            if (l_px.size > 0 and denom and denom > 0) else np.nan
        )
        rows.append(row)
    return pd.DataFrame(rows)


def run_quantification(gfp_img, mcherry_img, masks, membrane_thickness_px=5):
    """Full quantification step: masks -> per-GUV DataFrame."""
    membrane_masks, lumen_masks = make_membrane_lumen_masks(masks, membrane_thickness_px)
    df_gfp     = quantify(gfp_img,     masks, membrane_masks, lumen_masks, 'GFP')
    df_mcherry = quantify(mcherry_img, masks, membrane_masks, lumen_masks, 'mCherry')
    geom_cols  = ['label', 'centroid_x_px', 'centroid_y_px', 'area_px2', 'diameter_px']
    extra      = [c for c in df_mcherry.columns if c not in geom_cols]
    df = df_gfp.merge(df_mcherry[['label'] + extra], on='label')
    print(f'  Quantified {len(df)} GUVs')
    return df, membrane_masks, lumen_masks


print('Helper functions defined.')


---
### 2 · Load a single file for inspection

In [ ]:
FILE_INDEX = 1   # change to 1, 2, ... to inspect a different file

vsi_path = vsi_files[FILE_INDEX]
data, channel_names = read_vsi(vsi_path)


gfp_img, mcherry_img = split_channels(
    data, channel_names,
    gfp_idx=GFP_CHANNEL_IDX,
    mcherry_idx=MCHERRY_CHANNEL_IDX
)
seg_img = gfp_img if SEGMENTATION_CHANNEL_IDX == GFP_CHANNEL_IDX else mcherry_img
print(f'Segmentation channel: {"GFP" if SEGMENTATION_CHANNEL_IDX == GFP_CHANNEL_IDX else "mCherry"}')

### Channel sanity check
Run this once to visually confirm which index is GFP and which is mCherry.  
GFP will show bright membrane rings; mCherry will have a different spatial pattern.  
If they look swapped, flip `GFP_CHANNEL_IDX` and `MCHERRY_CHANNEL_IDX` in the config cell.

In [ ]:
fig, axes = plt.subplots(1, data.shape[0], figsize=(7 * data.shape[0], 6))
if data.shape[0] == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.imshow(normalise(data[i]), cmap='gray', vmin=0, vmax=255)
    tag = ''
    if i == GFP_CHANNEL_IDX:     tag = '  ->  assigned as GFP'
    if i == MCHERRY_CHANNEL_IDX: tag = '  ->  assigned as mCherry'
    ax.set_title(f'Channel index {i}\n"{channel_names[i]}"{tag}', fontsize=11)
    ax.axis('off')

plt.suptitle('Channel sanity check -- verify assignments look correct', fontsize=12)
plt.tight_layout()
plt.show()

### Quick colour preview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(normalise(gfp_img),     cmap='Greens', vmin=0, vmax=255)
axes[0].set_title(f'GFP  (channel index {GFP_CHANNEL_IDX})', fontsize=13)
axes[1].imshow(normalise(mcherry_img), cmap='Reds',   vmin=0, vmax=255)
axes[1].set_title(f'mCherry  (channel index {MCHERRY_CHANNEL_IDX})', fontsize=13)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

---
### 3 · Model comparison: cyto3 vs Cellpose-SAM *(archived — cpsam used by default)*

This section runs both models side-by-side in napari for manual comparison.
It is kept for reference but does not need to be run — the pipeline now defaults to Cellpose-SAM.
To re-enable, convert the cell below from Raw back to Code.


In [ ]:
CHOSEN_MODEL = 'cpsam'


---
### 4 · Inspect & manually correct masks in napari

A **standalone napari window** will open separately from VS Code.  
Layers:
- **GFP** — green, additive blend
- **mCherry** — red, additive blend  
- **GUV masks** — editable Labels layer

### Editing cheatsheet
| Action | How |
|---|---|
| Select a label | Alt-click on a GUV, or type its number in the Labels panel |
| Paint a new GUV | Pick the next unused label number, draw with brush |
| Erase a bad mask | Switch to label `0` (eraser), paint over it |
| Fill a closed outline | Bucket (fill) tool |
| Undo | Ctrl+Z |

### More editing notes
Modifying individual GUV masks
Each GUV has a unique integer label (1, 2, 3 …). To edit just one:

1. Pick its label — press L (pick mode) then click on the GUV. The active label spinner in the left panel snaps to that GUV's integer ID.
2. Switch to Brush and paint/fill only that label's pixels.
3. To delete a whole GUV: pick it, then press Fill and paint it with label 0, or use Ctrl+click in some napari versions to flood-fill to 0.
4. To split or reshape: erase part of the current GUV, then pick a new unused label number and paint the split-off region.
The label number shown in the spinner is the only thing that determines which GUV you're drawing — napari won't prevent you from painting one GUV over another, so always pick first.



**When done: close the napari window — the cell completes — then run the retrieval cell below.**

### 4a · Segmentation preview *(archived — run 4c for batch)*

Static preview of the single-file segmentation result. Kept for reference;
run **4c** (batch interactive loop) instead. To re-enable, convert the cell below from Raw back to Code.

### 4b · Verify membrane / lumen ring *(archived — run 4c for batch)*

Visual check for `MEMBRANE_THICKNESS_PX`. Kept for reference;
run **4c** to process files with the current setting. To re-enable, convert the cells below from Raw back to Code.

---
### 4c · Batch interactive processing

Assign a list of file indices and the pipeline will open each one in sequence:  
**load → segment → napari (manual correction) → quantify**.  
Results accumulate into `batch_df` and are visualised together in the following cell.

1. Edit `FILE_INDICES` below.
2. Run the cell — a napari window opens for the first file.
3. Correct the masks, then **close the napari window** to continue to the next file.
4. Repeat until all files are done, then run the plot cell.

In [ ]:
FILE_INDICES = [0,1]   # ← edit: list of indices into vsi_files

print('Files selected for batch processing:')
for _i, _idx in enumerate(FILE_INDICES):
    print(f'  [{_i}] index {_idx}: {vsi_files[_idx].name}')
print('\nAdjust FILE_INDICES above if needed, then run the next cell to start.')


In [ ]:
from matplotlib.patches import Patch as _Patch
from qtpy.QtWidgets import QApplication as _QApp
import time as _time

def _wait_for_close(viewer):
    """Pump the Qt event loop until the napari window is closed."""
    _app  = _QApp.instance()
    _qwin = viewer.window._qt_window
    while _qwin.isVisible():
        _app.processEvents()
        _time.sleep(0.05)


_batch_dfs      = []
_report_figs    = []   # (section_title, Path) pairs collected for the HTML report
_training_pairs = []   # (stem, gfp_array, masks_array) — consumed by Section 5-i

for _i, _idx in enumerate(FILE_INDICES):
    _vsi = vsi_files[_idx]
    print(f'\n{"─" * 60}')
    print(f'  [{_i + 1} / {len(FILE_INDICES)}]  {_vsi.name}')
    print(f'{"─" * 60}')

    # ── Load ──────────────────────────────────────────────────────────────
    _data, _ch = read_vsi(_vsi)
    _gfp, _mch = split_channels(
        _data, _ch,
        gfp_idx=GFP_CHANNEL_IDX,
        mcherry_idx=MCHERRY_CHANNEL_IDX,
    )
    _seg = _gfp if SEGMENTATION_CHANNEL_IDX == GFP_CHANNEL_IDX else _mch

    # ── Segment (using the model chosen in Section 3a) ───────────────────
    if CHOSEN_MODEL == 'cpsam':
        _masks = segment_guvs(
            _seg,
            diameter           = None,
            flow_threshold     = CP_FLOW_THRESHOLD,
            cellprob_threshold = 0.0,
            model_type         = 'cpsam',
            use_gpu            = CP_GPU,
            min_size           = CP_MIN_AREA_PX,
        )
    else:
        _masks = segment_guvs(
            _seg,
            diameter           = CP_DIAMETER,
            flow_threshold     = CP_FLOW_THRESHOLD,
            cellprob_threshold = CP_CELLPROB_THRESHOLD,
            model_type         = CP_MODEL,
            use_gpu            = CP_GPU,
            min_size           = CP_MIN_AREA_PX,
        )

    # ── Napari: manual mask correction ────────────────────────────────────
    _viewer = napari.Viewer(
        title=f'[{_idx}] {_vsi.name} — correct masks, then CLOSE this window to continue'
    )
    _viewer.add_image(normalise(_gfp) / 255.0, name='GFP',     colormap='green', blending='additive')
    _viewer.add_image(normalise(_mch) / 255.0, name='mCherry', colormap='red',   blending='additive')
    _lbl_layer = _viewer.add_labels(_masks, name='Masks')

    _wait_for_close(_viewer)   # pumps Qt events until the window is closed

    _masks_ok = _lbl_layer.data.astype(np.int32)

    if _masks_ok.max() == 0:
        print(f'  [skip] No masks remaining for {_vsi.name} — skipping quantification.')
        continue

    _training_pairs.append((_vsi.stem, _gfp, _masks_ok))

    # ── Fig 1: segmentation preview ───────────────────────────────────────
    _gfp_n = normalise(_gfp) / 255.0
    _mch_n = normalise(_mch) / 255.0
    _fig1, _ax1 = plt.subplots(1, 4, figsize=(24, 6))
    _ax1[0].imshow(_gfp_n, cmap='Greens', vmin=0, vmax=1);  _ax1[0].set_title('GFP (normalised)');     _ax1[0].axis('off')
    _ax1[1].imshow(_mch_n, cmap='Reds',   vmin=0, vmax=1);  _ax1[1].set_title('mCherry (normalised)'); _ax1[1].axis('off')
    _ax1[2].imshow(_masks_ok, cmap='nipy_spectral', interpolation='nearest')
    _ax1[2].set_title(f'Masks ({int(_masks_ok.max())} GUVs)'); _ax1[2].axis('off')
    _rgb_overlay = np.stack([_mch_n, _gfp_n, np.zeros_like(_gfp_n)], axis=-1).clip(0, 1)
    _ax1[3].imshow(_rgb_overlay)
    _ax1[3].imshow(_masks_ok, cmap='nipy_spectral', interpolation='nearest', alpha=0.3)
    _ax1[3].set_title(f'GFP + mCherry + Masks overlay'); _ax1[3].axis('off')
    for _lid in np.unique(_masks_ok):
        if _lid == 0:
            continue
        _ys, _xs = np.where(_masks_ok == _lid)
        _cx, _cy = float(_xs.mean()), float(_ys.mean())
        for _a1 in [_ax1[2], _ax1[3]]:
            _a1.text(_cx, _cy, str(_lid),
                color='white', fontsize=12, fontweight='bold', ha='center', va='center',
                bbox=dict(facecolor='black', alpha=0.5, edgecolor='none', pad=1))
    _fig1.suptitle(f'Segmentation — {_vsi.stem}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    _seg_path = OUTPUT_DIR / f'{_vsi.stem}_segmentation_preview.png'
    _fig1.savefig(_seg_path, dpi=150, bbox_inches='tight')
    _report_figs.append((f'{_vsi.stem}  ·  segmentation', _seg_path))
    plt.show()

    # ── Fig 2: ring segmentation panel ────────────────────────────────────
    _mem_prev, _lum_prev = make_membrane_lumen_masks(_masks_ok, MEMBRANE_THICKNESS_PX)
    _is_mem = np.zeros(_masks_ok.shape, bool)
    _is_lum = np.zeros(_masks_ok.shape, bool)
    for _mpx in _mem_prev.values(): _is_mem |= _mpx
    for _lpx in _lum_prev.values(): _is_lum |= _lpx
    _ORANGE = [1.00, 0.65, 0.00, 1.0]
    _BLUE   = [0.39, 0.58, 0.93, 1.0]
    _ring_rgba = np.zeros((*_masks_ok.shape, 4), dtype=np.float32)
    _ring_rgba[_is_mem] = _ORANGE
    _ring_rgba[_is_lum] = _BLUE
    _ring_semi = _ring_rgba.copy()
    _ring_semi[..., 3] *= 0.55
    _lgd_r = [_Patch(facecolor=_ORANGE[:3], label='Membrane'),
              _Patch(facecolor=_BLUE[:3],   label='Lumen')]
    _fig2, _ax2 = plt.subplots(1, 4, figsize=(22, 6))
    _ax2[0].imshow(_masks_ok, cmap='nipy_spectral', interpolation='nearest')
    _ax2[0].set_title(f'Masks ({int(_masks_ok.max())} GUVs)', fontsize=11); _ax2[0].axis('off')
    _ax2[1].imshow(np.zeros((*_masks_ok.shape, 3), dtype=np.uint8))
    _ax2[1].imshow(_ring_rgba, interpolation='nearest')
    _ax2[1].set_title(f'Ring annotation  ({MEMBRANE_THICKNESS_PX} px)', fontsize=11)
    _ax2[1].legend(handles=_lgd_r, loc='lower left', fontsize=8, framealpha=0.6); _ax2[1].axis('off')
    _ax2[2].imshow(_gfp_n, cmap='Greens', vmin=0, vmax=1)
    _ax2[2].imshow(_ring_semi, interpolation='nearest')
    _ax2[2].set_title('GFP + ring overlay', fontsize=11)
    _ax2[2].legend(handles=_lgd_r, loc='lower left', fontsize=8, framealpha=0.6); _ax2[2].axis('off')
    _ax2[3].imshow(_mch_n, cmap='Reds', vmin=0, vmax=1)
    _ax2[3].imshow(_ring_semi, interpolation='nearest')
    _ax2[3].set_title('mCherry + ring overlay', fontsize=11)
    _ax2[3].legend(handles=_lgd_r, loc='lower left', fontsize=8, framealpha=0.6); _ax2[3].axis('off')
    _fig2.suptitle(f'Ring segmentation — {_vsi.stem}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    _ring_path = OUTPUT_DIR / f'{_vsi.stem}_ring_preview.png'
    _fig2.savefig(_ring_path, dpi=150, bbox_inches='tight')
    _report_figs.append((f'{_vsi.stem}  ·  ring segmentation', _ring_path))
    plt.show()

    # ── Quantify ──────────────────────────────────────────────────────────
    _df, _, _ = run_quantification(_gfp, _mch, _masks_ok, MEMBRANE_THICKNESS_PX)
    _df.insert(0, 'file_idx', _idx)
    _df.insert(1, 'file',     _vsi.stem)
    # F_bulk: mode of unmasked pixels (external phase dominates histogram peak)
    def _bulk_mode(img, mask_ok, bins=512):
        outside = img[mask_ok == 0].ravel()
        counts, edges = np.histogram(outside, bins=bins)
        peak = np.argmax(counts)
        return 0.5 * (edges[peak] + edges[peak + 1])
    _gfp_bulk = _bulk_mode(_gfp, _masks_ok)
    _mch_bulk = _bulk_mode(_mch, _masks_ok)
    # Background-subtracted means — same intensity units, valid for stoichiometric ratio
    _df['GFP_lumen_bgsub']         = (_df['GFP_lumen_mean']     - _gfp_bulk).clip(0)
    _df['mCherry_lumen_bgsub']     = (_df['mCherry_lumen_mean'] - _mch_bulk).clip(0)
    _df['GFP_mCherry_lumen_ratio'] = _df['GFP_lumen_bgsub'] / (_df['mCherry_lumen_bgsub'] + 1e-9)
    print(_df)

    # ── Fig 3: membrane / lumen ratio scatter ────────────────────────
    _gfp_ratio = _df['GFP_mem_lumen_ratio']
    _mch_ratio = _df['mCherry_mem_lumen_ratio']
    _fig3, _ax3 = plt.subplots(figsize=(7, 6))
    _ax3.scatter(
        _gfp_ratio, _mch_ratio,
        alpha=0.7, edgecolors='k', linewidths=0.5, c='dodgerblue', s=60,
    )
    _ax3.set_xlabel('GFP Membrane / Lumen Intensity Ratio')
    _ax3.set_ylabel('mCherry Membrane / Lumen Intensity Ratio')
    _ax3.set_title(f'Membrane / Lumen Ratio — {_vsi.stem}', fontsize=11)
    _ax3.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    _mem_lum_path = OUTPUT_DIR / f'{_vsi.stem}_ratio_scatter.png'
    _fig3.savefig(_mem_lum_path, dpi=150, bbox_inches='tight')
    _report_figs.append((f'{_vsi.stem}  ·  membrane / lumen ratio', _mem_lum_path))
    plt.show()

    # ── Fig 4: per-file lumen GFP / mCherry ratio ────────────────────────
    _glum = _df['GFP_lumen_bgsub']
    _mlum = _df['mCherry_lumen_bgsub']
    _rat  = _df['GFP_mCherry_lumen_ratio']
    _fig4, _fa = plt.subplots(figsize=(7, 6))
    _sc = _fa.scatter(_mlum, _glum, c=_rat, cmap='RdBu_r', vmin=0, vmax=2,
                      alpha=0.8, edgecolors='k', linewidths=0.4, s=70)
    plt.colorbar(_sc, ax=_fa, label='GFP / mCherry lumen intensity ratio (bg-subtracted)')
    _lim3 = max(_mlum.max(), _glum.max()) * 1.05
    _fa.plot([0, _lim3], [0, _lim3], '--', color='grey', lw=1, label='1:1')
    _fa.set_xlim(0, _lim3); _fa.set_ylim(0, _lim3)
    _fa.set_xlabel('mCherry lumen intensity (bg-subtracted, a.u.)')
    _fa.set_ylabel('GFP lumen intensity (bg-subtracted, a.u.)')
    _fa.set_title(f'Lumen fluorescence ratio — {_vsi.stem}', fontsize=11)
    _fa.legend(fontsize=9); _fa.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    _lum_path = OUTPUT_DIR / f'{_vsi.stem}_lumen_ratio.png'
    _fig4.savefig(_lum_path, dpi=150, bbox_inches='tight')
    _report_figs.append((f'{_vsi.stem}  ·  lumen GFP / mCherry ratio', _lum_path))
    plt.show()

    _out = OUTPUT_DIR / f'{_vsi.stem}_measurements.csv'
    _df.to_csv(_out, index=False)
    print(f'  Saved -> {_out}')
    _batch_dfs.append(_df)

batch_df = pd.concat(_batch_dfs, ignore_index=True)
_batch_csv = OUTPUT_DIR / 'batch_interactive_measurements.csv'
batch_df.to_csv(_batch_csv, index=False)
print(f'\nAll done — {len(batch_df)} GUVs across {len(FILE_INDICES)} files.')
print(f'Combined CSV -> {_batch_csv}')

In [ ]:
import seaborn as _sns

_files   = list(batch_df['file'].unique())
_palette = plt.cm.tab10(np.linspace(0, 0.9, max(len(_files), 2)))[:len(_files)]
_cmap_b  = dict(zip(_files, _palette))

fig, (ax3, ax_vln, ax_diam) = plt.subplots(1, 3, figsize=(24, 8))

for _fname, _grp in batch_df.groupby('file'):
    _col = _cmap_b[_fname]
    ax3.scatter(
        _grp['GFP_mem_lumen_ratio'], _grp['mCherry_mem_lumen_ratio'],
        color=_col, label=_fname, alpha=0.75,
        edgecolors='k', linewidths=0.3, s=60,
    )
    ax_diam.scatter(
        _grp['diameter_px'], _grp['mCherry_mem_lumen_ratio'],
        color=_col, label=_fname, alpha=0.75,
        edgecolors='k', linewidths=0.3, s=60,
    )

ax3.axvline(1.0, color='grey', linestyle=':', lw=1.5)
ax3.axhline(1.0, color='grey', linestyle=':', lw=1.5)

# ── Quadrant counts ───────────────────────────────────────────────────────────
_gx = batch_df['GFP_mem_lumen_ratio']
_my = batch_df['mCherry_mem_lumen_ratio']
_xlim = ax3.get_xlim(); _ylim = ax3.get_ylim()
_pad = 0.04  # fractional inset from the reference lines
_xlo, _xhi = _gx.min(), _gx.max()
_ylo, _yhi = _my.min(), _my.max()
_kw_q = dict(fontsize=11, color='#444', transform=ax3.transAxes,
             bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6, ec='none'))
for (_xmin, _xmax, _ymin, _ymax), (_ax, _ay, _ha, _va) in [
    ((1, _xhi, 1, _yhi), (0.97, 0.97, 'right', 'top')),
    ((1, _xhi, _ylo, 1), (0.97, 0.03, 'right', 'bottom')),
    ((_xlo, 1, 1, _yhi), (0.03, 0.97, 'left',  'top')),
    ((_xlo, 1, _ylo, 1), (0.03, 0.03, 'left',  'bottom')),
]:
    _n = int(((_gx > _xmin) & (_gx <= _xmax) & (_my > _ymin) & (_my <= _ymax)).sum())
    ax3.text(_ax, _ay, f'n={_n}', ha=_ha, va=_va, **_kw_q)

ax3.set_xlabel('GFP Membrane / Lumen Ratio', fontsize=15)
ax3.set_ylabel('mCherry Membrane / Lumen Ratio', fontsize=15)
ax3.tick_params(axis='both', labelsize=15)
ax3.set_title('Membrane / Lumen Ratio', fontsize=15)
ax3.legend(fontsize=12, title='File', framealpha=0.6, bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0)
ax3.spines[['top', 'right']].set_visible(False)

_melt = batch_df[['file', 'GFP_mem_lumen_ratio', 'mCherry_mem_lumen_ratio']].melt(
    id_vars='file', var_name='channel', value_name='ratio'
)
_melt['channel'] = _melt['channel'].map({
    'GFP_mem_lumen_ratio':     'GFP',
    'mCherry_mem_lumen_ratio': 'mCherry',
})
_sns.violinplot(
    data=_melt, x='channel', y='ratio', fill=False,
    palette={'GFP': 'mediumseagreen', 'mCherry': 'tomato'},
    inner='box', cut=0, ax=ax_vln,
)
ax_vln.axhline(1.0, color='grey', linestyle=':', lw=1.5)
ax_vln.set_xlabel('Channel', fontsize=15)
ax_vln.set_ylabel('Membrane / Lumen Ratio', fontsize=15)
ax_vln.tick_params(axis='both', labelsize=15)
ax_vln.set_title('Membrane / Lumen Ratio Distribution', fontsize=15)
ax_vln.spines[['top', 'right']].set_visible(False)

ax_diam.axhline(1.0, color='grey', linestyle=':', lw=1.5)
ax_diam.set_xlabel('Diameter (px)', fontsize=15)
ax_diam.set_ylabel('mCherry Membrane / Lumen Ratio', fontsize=15)
ax_diam.tick_params(axis='both', labelsize=15)
ax_diam.set_title('mCherry Mem/Lumen Ratio vs Size', fontsize=15)
ax_diam.legend(fontsize=12, title='File', framealpha=0.6)
ax_diam.spines[['top', 'right']].set_visible(False)

fig.suptitle(
    f'Batch interactive results  ({len(_files)} files, {len(batch_df)} GUVs total)',
    fontsize=13, fontweight='bold',
)
plt.tight_layout()
_out = OUTPUT_DIR / 'batch_interactive_lumen_ratio.png'
plt.savefig(_out, dpi=300, bbox_inches='tight')
print(f'Saved -> {_out}')
plt.show()
_report_figs.append(('Combined results  ·  membrane / lumen ratio', _out))


In [ ]:
# ── HTML report — embeds every figure as base64 ───────────────────────────────
import base64

def _b64_img(path):
    with open(path, 'rb') as _f:
        return base64.b64encode(_f.read()).decode()

# Group figure entries by file stem for section headers
_seen_stems = []
_html_body  = []
_current_stem = None

for _title, _path in _report_figs:
    _stem = _title.split('  ·  ')[0].strip()
    if _stem != _current_stem:
        if _current_stem is not None:
            _html_body.append('  </section>\n')
        _html_body.append(f'  <section>\n    <h2>{_stem}</h2>\n')
        _current_stem = _stem
    _subtitle = _title.split('  ·  ', 1)[-1] if '  ·  ' in _title else _title
    _b64 = _b64_img(_path)
    _html_body.append(
        f'    <figure>\n'
        f'      <figcaption>{_subtitle}</figcaption>\n'
        f'      <img src="data:image/png;base64,{_b64}"\n'
        f'           style="max-width:100%;border:1px solid #ddd;border-radius:4px;"/>\n'
        f'    </figure>\n'
    )
if _current_stem is not None:
    _html_body.append('  </section>\n')

_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8"/>
  <title>GUV Batch Analysis Report</title>
  <style>
    body      {{ font-family: Arial, sans-serif; max-width: 1500px;
                margin: auto; padding: 24px; background: #fafafa; }}
    h1        {{ color: #222; border-bottom: 3px solid #888; padding-bottom: 6px; }}
    h2        {{ color: #444; margin-top: 48px; border-left: 4px solid #888;
                padding-left: 10px; }}
    p.meta    {{ color: #666; font-size: 0.9em; }}
    section   {{ margin-bottom: 20px; }}
    figure    {{ margin: 16px 0; }}
    figcaption{{ font-size: 0.85em; color: #555; margin-bottom: 6px;
                font-style: italic; }}
  </style>
</head>
<body>
  <h1>GUV Batch Analysis Report</h1>
  <p class="meta">{len(FILE_INDICES)} files &nbsp;&middot;&nbsp;
  {len(batch_df)} GUVs total &nbsp;&middot;&nbsp;
  {len(_report_figs)} figures</p>
{"".join(_html_body)}
</body>
</html>"""

_html_path = OUTPUT_DIR / 'batch_interactive_report.html'
_html_path.write_text(_html, encoding='utf-8')
print(f'Saved HTML report -> {_html_path}')

---
> **Pipeline complete.** If you processed all your files interactively in Section 4c, you are done — per-file CSVs and figures are in `OUTPUT_DIR`, and the combined CSV is `batch_interactive_measurements.csv`.
>
> **Sections 5 and 6 are optional** and only needed if you want to:
> - **Section 5** — fine-tune Cellpose on your own corrected masks to improve segmentation on unusual GUV morphologies
> - **Section 6** — run additional files headlessly (no napari, no manual correction) using the parameters already tuned
>
> If neither applies to your workflow, you can stop here.

--- 
## 5 · Fine-tune Cellpose on your own GUV masks

Use the napari-corrected masks from Section 4 as training data to adapt Cellpose to your specific GUVs.
Run **5-i** after each correction session to accumulate pairs, then **5-ii** once you have ≥ 20 pairs.

In [ ]:
# ── 5-i · Save training pairs ──────────────────────────────────────────────
# Run this after Section 4c to save all napari-corrected image/mask pairs as TIFFs.
# Uses _training_pairs accumulated during the 4c loop — no need to re-load files.

TRAINING_DIR = OUTPUT_DIR / 'training_data'
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

if not _training_pairs:
    print('No training pairs available — run Section 4c first.')
else:
    for _stem, _img, _msk in _training_pairs:
        img_path  = TRAINING_DIR / f'{_stem}_img.tif'
        mask_path = TRAINING_DIR / f'{_stem}_masks.tif'
        tifffile.imwrite(img_path,  _img.astype(np.uint16))
        tifffile.imwrite(mask_path, _msk.astype(np.uint16))
        print(f'Saved: {img_path.name}')
        print(f'Saved: {mask_path.name}')

    existing_pairs = sorted(TRAINING_DIR.glob('*_img.tif'))
    n_pairs = len(existing_pairs)
    print(f'\nTraining pairs accumulated: {n_pairs}')
    if n_pairs < 20:
        print(f'  Only {n_pairs} pair(s) — aim for >= 20 before fine-tuning.')

In [ ]:
# ── 5-ii · Fine-tune Cellpose ──────────────────────────────────────────────
from cellpose import train

# ── user-editable parameters ────────────────────────────────────────────────
START_MODEL    = 'cyto3'   # 'cyto3' or 'cpsam'
N_EPOCHS       = 100
LEARNING_RATE  = 0.1
WEIGHT_DECAY   = 1e-5

FINETUNED_MODEL_PATH = OUTPUT_DIR / 'cellpose_GUV_model'
TRAINING_DIR         = OUTPUT_DIR / 'training_data'

# ── load image / mask pairs ─────────────────────────────────────────────────
img_paths  = sorted(TRAINING_DIR.glob('*_img.tif'))
mask_paths = [p.with_name(p.name.replace('_img.tif', '_masks.tif')) for p in img_paths]

if len(img_paths) == 0:
    raise FileNotFoundError(f'No training pairs found in {TRAINING_DIR}. Run 5b-i first.')

train_images = [tifffile.imread(p).astype(np.float32) for p in img_paths]
train_masks  = [tifffile.imread(p).astype(np.int32)   for p in mask_paths]
print(f'Loaded {len(train_images)} training pair(s) from {TRAINING_DIR}')

# ── fine-tune ───────────────────────────────────────────────────────────────
model_save_path = train.train_seg(
    train_images,
    train_masks,
    channels   = [0, 0],
    normalize  = True,
    pretrained_model = START_MODEL,
    n_epochs         = N_EPOCHS,
    learning_rate    = LEARNING_RATE,
    weight_decay     = WEIGHT_DECAY,
    model_name       = str(FINETUNED_MODEL_PATH),
)
print(f'Fine-tuned model saved -> {model_save_path}')
FINETUNED_MODEL_PATH = model_save_path   # update in case Cellpose appended a suffix


In [ ]:
# ── 5-iii · Switch model ────────────────────────────────────────────────────
# Set USE_FINETUNED_MODEL = True to use the fine-tuned model from 5b-ii;
# False keeps the original cyto3 behaviour.

USE_FINETUNED_MODEL = True   # <-- toggle here

if USE_FINETUNED_MODEL:
    _ft_path = str(FINETUNED_MODEL_PATH)
    def segment_guvs(gfp_img, **kwargs):
        """Fine-tuned Cellpose model for GUVs.  **kwargs absorbs any extra args."""
        from cellpose import models as _cp_models
        _model = _cp_models.CellposeModel(pretrained_model=_ft_path)
        _masks, _, _ = _model.eval(
            normalise(gfp_img),
            channels          = [0, 0],
            normalize         = False,   # already normalised
            flow_threshold    = kwargs.get('flow_threshold',    CP_FLOW_THRESHOLD),
            cellprob_threshold= kwargs.get('cellprob_threshold', CP_CELLPROB_THRESHOLD),
            diameter          = kwargs.get('diameter',           CP_DIAMETER),
            do_3D             = False,
        )
        _n = int(_masks.max())
        print(f'  fine-tuned -> {_n} GUVs detected')
        return _masks.astype(np.int32)
    print(f'segment_guvs -> fine-tuned model ({_ft_path})')
else:
    def segment_guvs(gfp_img, diameter=CP_DIAMETER, flow_threshold=CP_FLOW_THRESHOLD,
                     cellprob_threshold=CP_CELLPROB_THRESHOLD, model_type='cyto3',
                     use_gpu=CP_GPU, min_size=CP_MIN_AREA_PX, **kwargs):
        """Original cyto3 implementation.  **kwargs absorbs any extra args."""
        from cellpose import models as _cp_models
        _model = _cp_models.CellposeModel(gpu=use_gpu, pretrained_model=model_type)
        _masks, _, _ = _model.eval(
            normalise(gfp_img),
            channels          = [0, 0],
            diameter          = diameter,
            flow_threshold    = flow_threshold,
            cellprob_threshold= cellprob_threshold,
            min_size          = min_size,
            do_3D             = False,
        )
        _n = int(_masks.max())
        print(f'  cyto3 -> {_n} GUVs detected')
        return _masks.astype(np.int32)
    print('segment_guvs -> cyto3 (original)')


In [ ]:
# ── 5-iv · Evaluate (optional) ─────────────────────────────────────────────
# Load a held-out image + mask pair and compare cyto3 vs the fine-tuned model.
import napari
from cellpose import models as _cp_models, metrics

# ── user-editable: paths to a held-out pair ─────────────────────────────────
HELD_OUT_IMG_PATH  = TRAINING_DIR / 'CHANGE_ME_img.tif'
HELD_OUT_MASK_PATH = TRAINING_DIR / 'CHANGE_ME_masks.tif'

if not HELD_OUT_IMG_PATH.exists():
    raise FileNotFoundError(f'Set HELD_OUT_IMG_PATH to a real file (got {HELD_OUT_IMG_PATH})')

held_img  = tifffile.imread(HELD_OUT_IMG_PATH).astype(np.float32)
held_mask = tifffile.imread(HELD_OUT_MASK_PATH).astype(np.int32)
print(f'Held-out image : {HELD_OUT_IMG_PATH.name}  shape={held_img.shape}')
print(f'Held-out mask  : {HELD_OUT_MASK_PATH.name}  GUVs={int(held_mask.max())}')

# ── cyto3 prediction ────────────────────────────────────────────────────────
_model_cyto3 = _cp_models.CellposeModel(pretrained_model='cyto3')
_pred_cyto3, _, _ = _model_cyto3.eval(
    normalise(held_img), channels=[0, 0],
    diameter=CP_DIAMETER, flow_threshold=CP_FLOW_THRESHOLD,
    cellprob_threshold=CP_CELLPROB_THRESHOLD, do_3D=False,
)
_pred_cyto3 = _pred_cyto3.astype(np.int32)

# ── fine-tuned model prediction ─────────────────────────────────────────────
_model_ft = _cp_models.CellposeModel(pretrained_model=str(FINETUNED_MODEL_PATH))
_pred_ft, _, _ = _model_ft.eval(
    normalise(held_img), channels=[0, 0],
    diameter=CP_DIAMETER, flow_threshold=CP_FLOW_THRESHOLD,
    cellprob_threshold=CP_CELLPROB_THRESHOLD, do_3D=False,
)
_pred_ft = _pred_ft.astype(np.int32)

# ── Average Precision @ IoU=0.5 ──────────────────────────────────────────────
_ap_cyto3, _, _ = metrics.average_precision([held_mask], [_pred_cyto3], threshold=[0.5])
_ap_ft,    _, _ = metrics.average_precision([held_mask], [_pred_ft],    threshold=[0.5])
print(f'\nAverage Precision @ IoU=0.5:')
print(f'  cyto3       : {float(_ap_cyto3[0]):.3f}')
print(f'  fine-tuned  : {float(_ap_ft[0]):.3f}')

# ── napari: toggle between both mask layers ───────────────────────────────────
_img_norm = normalise(held_img) / 255.0
viewer = napari.Viewer(title='Evaluation: cyto3 vs fine-tuned')
viewer.add_image(_img_norm, name='GFP (held-out)', colormap='green')
viewer.add_labels(_pred_cyto3, name='cyto3 prediction')
viewer.add_labels(_pred_ft,    name='fine-tuned prediction', visible=False)  # hidden by default
viewer.add_labels(held_mask,   name='ground truth',          visible=False)
napari.run()   # blocks until the window is closed


---
## 6 · Batch process remaining files

Runs headlessly (no napari) using the parameters tuned above.  
Go back to Sections 2–4 for any individual file that needs manual inspection.

In [ ]:
all_dfs = []

# Skip files already processed interactively in Section 4c
try:
    _done_stems = set(batch_df['file'].unique())
    if _done_stems:
        print(f'Skipping {len(_done_stems)} file(s) already processed in 4c: {", ".join(sorted(_done_stems))}')
except NameError:
    _done_stems = set()

for vsi in vsi_files:
    if vsi.stem in _done_stems:
        continue
    print(f'\n-- {vsi.name} --')
    try:
        _data, _cnames = read_vsi(vsi)
        _gfp, _mcherry = split_channels(
            _data, _cnames,
            gfp_idx=GFP_CHANNEL_IDX,
            mcherry_idx=MCHERRY_CHANNEL_IDX,
        )
        _seg = _gfp if SEGMENTATION_CHANNEL_IDX == GFP_CHANNEL_IDX else _mcherry
        if CHOSEN_MODEL == 'cpsam':
            _masks = segment_guvs(
                _seg,
                diameter=None, flow_threshold=CP_FLOW_THRESHOLD,
                cellprob_threshold=0.0, model_type='cpsam',
                use_gpu=CP_GPU, min_size=CP_MIN_AREA_PX,
            )
        else:
            _masks = segment_guvs(
                _seg,
                diameter=CP_DIAMETER, flow_threshold=CP_FLOW_THRESHOLD,
                cellprob_threshold=CP_CELLPROB_THRESHOLD, model_type=CP_MODEL,
                use_gpu=CP_GPU, min_size=CP_MIN_AREA_PX,
            )
        if _masks.max() == 0:
            print('  WARNING: No GUVs detected — skipping')
            continue

        _df, _, _ = run_quantification(_gfp, _mcherry, _masks, MEMBRANE_THICKNESS_PX)
        _df.insert(0, 'file', vsi.stem)

        # Background subtraction (same method as Section 4c)
        def _bulk_mode(img, mask_ok, bins=512):
            outside = img[mask_ok == 0].ravel()
            counts, edges = np.histogram(outside, bins=bins)
            peak = np.argmax(counts)
            return 0.5 * (edges[peak] + edges[peak + 1])
        _gfp_bulk = _bulk_mode(_gfp, _masks)
        _mch_bulk = _bulk_mode(_mcherry, _masks)
        _df['GFP_lumen_bgsub']         = (_df['GFP_lumen_mean']     - _gfp_bulk).clip(0)
        _df['mCherry_lumen_bgsub']     = (_df['mCherry_lumen_mean'] - _mch_bulk).clip(0)
        _df['GFP_mCherry_lumen_ratio'] = _df['GFP_lumen_bgsub'] / (_df['mCherry_lumen_bgsub'] + 1e-9)

        _df.to_csv(OUTPUT_DIR / f'{vsi.stem}_measurements.csv', index=False)
        print(f'  Saved {vsi.stem}_measurements.csv')
        all_dfs.append(_df)

    except AssertionError as e:
        print(f'  CHANNEL STRUCTURE ERROR: {e}')
    except Exception as e:
        print(f'  ERROR: {e}')

print('\nBatch complete.')

---
## 7 · Combined results

Merges results from Section 4c (interactive batch) and/or Section 6 (headless batch) into a single DataFrame, saves `all_GUV_measurements.csv`, and produces per-file boxplots.

In [ ]:
_frames = []
try:
    _frames.append(batch_df)
    print(f'Interactive results (4c) : {len(batch_df)} GUVs across {batch_df["file"].nunique()} file(s)')
except NameError:
    print('No interactive results (Section 4c not run)')

if all_dfs:
    _headless_df = pd.concat(all_dfs, ignore_index=True)
    _frames.append(_headless_df)
    print(f'Headless results (6)     : {len(_headless_df)} GUVs across {_headless_df["file"].nunique()} file(s)')
else:
    print('No headless results (Section 6 not run or produced no output)')

if not _frames:
    print('\nNo results to combine — run Section 4c and/or Section 6 first.')
else:
    combined = pd.concat(_frames, ignore_index=True)
    combined_path = OUTPUT_DIR / 'all_GUV_measurements.csv'
    combined.to_csv(combined_path, index=False)
    print(f'\nCombined: {len(combined)} GUVs across {combined["file"].nunique()} file(s)')
    print(f'Saved -> {combined_path}')
    display(combined.groupby('file')[[
        'GFP_membrane_mean', 'mCherry_membrane_mean',
        'GFP_mem_lumen_ratio', 'mCherry_mem_lumen_ratio'
    ]].mean().round(2))

In [ ]:
try:
    import seaborn as sns
except ImportError:
    print('pip install seaborn  for this plot'); raise

metrics = [
    ('GFP_membrane_mean',      'GFP membrane'),
    ('mCherry_membrane_mean',  'mCherry membrane'),
    ('GFP_mem_lumen_ratio',    'GFP mem/lumen ratio'),
    ('mCherry_mem_lumen_ratio','mCherry mem/lumen ratio'),
]
available_m = [(c, t) for c, t in metrics if c in combined.columns]

fig, axes = plt.subplots(1, len(available_m), figsize=(5 * len(available_m), 5))
if len(available_m) == 1:
    axes = [axes]

for ax, (col, title) in zip(axes, available_m):
    sns.boxplot(data=combined,  x='file', y=col, ax=ax, palette='Set2',
                width=0.5, fliersize=3)
    sns.stripplot(data=combined, x='file', y=col, ax=ax,
                  color='k', alpha=0.3, size=3, jitter=True)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=35)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'all_files_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()